# Notebook 1 · Create the Main Mapper Skeleton
## 0 · Data Preparation

This notebook ingests the raw CSV (features as columns and participants as rows), removes redundant features (correlation + VIF), standardizes the numerical columns, and stores the tidy tables under `processed/`.  All downstream notebooks consume the files produced here, so ensure the configuration matches your dataset before running every cell.

## 1. Environment check

Run the next cell once per session to import all functions.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
import kmapper as km
from sklearn.cluster import DBSCAN
from kmapper import jupyter
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import sklearn
import matplotlib.colors as mcolors
import json
import networkx as nx
import plotly.graph_objects as go
import plotly.colors as pc
import plotly.io as pio
import ast
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import Normalize
from statsmodels.stats.outliers_influence import variance_inflation_factor


## 2. Import Raw Data and Standardize Features

Update 'df' directory and define 'features' for standardization process

In [ ]:
df = pd.read_csv("Directory_to_your_data_file.csv") #Load your data file here
features = df.iloc[:, 2:].values  # Choose which columns to use as features
scaler = StandardScaler() #Load StandardScaler
features_scaled = scaler.fit_transform(features) #Standardize features using StandardScaler

## 3. (Optional) Correlation and VIF analysis

This step is optional but recommended. You can remove redundant and correlated features. Make sure to change your output variable accordingly.

In [ ]:
corr_matrix = features.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.98)]
features_corr_filtered = features.drop(columns=to_drop)
features = scaler.fit_transform(features_corr_filtered)

## 4. (Optional) coloring your classess for early visualization

In this optional stage, you can color your classess for early visualization. You must have a class column in your original 'df'. Use 'condition_mapping' and assign '2' to the class of interest and '1' to the rest for coloring the class of interest only.

In [ ]:
condition_labels = df["Class"].values  # Ensure this column exists in df
condition_mapping = {"CVA": 2, "HC": 1, "PD": 1, "LL": 1}  #to only show CVA vs others

numeric_labels = np.array([condition_mapping[label] for label in condition_labels])
numeric_labels = numeric_labels.reshape(-1, 1) 
numeric_labels = numeric_labels / numeric_labels.max()

plotly_colorscale = [
    [0.0, "rgb(255,255,255)"],  
    [0.5, "rgb(255,125,125)"], 
    [1.0, "rgb(255,0,0)"],
]


## 5. Kepler Mapper

This is the main core code for the Mapper algorithm. You can change the Dimension Reduction function (TSNE, UMAP, PCA, etc) and its parameters as well as the four main parameters of the mapper.

In [ ]:
mapper = km.KeplerMapper(verbose = 2)
#projected_data = mapper.fit_transform(features_scaled, projection=sklearn.manifold.TSNE())
projected_data = mapper.fit_transform(features_scaled, projection=sklearn.manifold.TSNE(n_components=2, max_iter=2000, random_state=42, perplexity=40))
graph = mapper.map(
    projected_data,
    clusterer=DBSCAN(eps=5, min_samples=2),
    cover=km.Cover(n_cubes=8, perc_overlap=0.50),
)

## 6. Visualize and Save your Mapper as an HTML File

Please change the 'path_html' to the location you want to save the html figure of the Mapper. Use 'color_function_name' and 'custom_tooltip' and 'colorscale' if you have done step 4, otherwise disable those lines.

In [ ]:
mapper.visualize(
    graph,
    path_html=r"Y:\LabMembers\S Daneshgar\P2C\TDA\Results\3-18-2025 Sajjad\Interactive mapper\cube8-perc50\cube8-perc50.html",
    color_values=numeric_labels.flatten(),  # Must be 1D
    color_function_name=["Conditions"],  # Label for color function
    custom_tooltips=condition_labels,  # Show original condition names on hover
    colorscale=plotly_colorscale
)
print("Mapper graph visualization saved")

## 7. Prepare next dataframes for future use

This block creates three dataframes for you. These three dataframes contain information about nodes and edges of the Mapper for visualization purposes.

In [ ]:
node_data = []
for node_id, sample_indices in graph["nodes"].items():
    for sample in sample_indices:
        node_data.append({"node": node_id, "sample_index": sample})
df_nodes = pd.DataFrame(node_data)

edges = []
for node1, connections in graph["links"].items():
    for node2 in connections:
        edges.append({"node_1": node1, "node_2": node2})
df_edges = pd.DataFrame(edges)
df2 = df
df_nodes['Cube'] = df_nodes['node'].str.extract(r'cube(\d+)')
df2['node_numbers'] = [[] for _ in range(len(df2))]
for _, row in df_nodes.iterrows():
    instance_idx = row.iloc[1]
    df2.at[instance_idx, 'node_numbers'] = df2.at[instance_idx, 'node_numbers'] + [row['Cube']]
df2['node_numbers'] = df2['node_numbers'].apply(lambda x: list(set(x)))
dfmapper = pd.DataFrame(df_nodes['node'].unique(), columns=['Nodes'])
dfmapper['Edges'] = [[] for _ in range(len(dfmapper))]
edges_dict = {}
for _, row in df_edges.iterrows():
    node1, node2 = row['node_1'], row['node_2']

    if node1 in edges_dict:
        edges_dict[node1].append(node2)
    else:
        edges_dict[node1] = [node2]

    if node2 in edges_dict:
        edges_dict[node2].append(node1)
    else:
        edges_dict[node2] = [node1]
dfmapper['Edges'] = dfmapper['Nodes'].map(lambda node: edges_dict.get(node, []))



## 8. Save the new dataframes

Please set the saving directory and save the new three dataframes.

In [ ]:
df2.to_csv()
df_nodes.to_csv()
df_edges.to_csv()
dfmapper.to_csv()

## Continue to the next file "002_PostHoc_Mapper"